# Diagnose, Fix, Brief · The Meridian Pilot

A client's AI support agent is failing and they think it's the model. Your job is to find what's actually wrong, fix it, and prove the fix with numbers the client can take to their leadership. You can change anything around the model. You can't change the model itself.

## Before you start

You'll work this the way you'd work a stalled client system in real life: poke it, watch what it does, form a theory, change one thing, and see if the number moves.

About that number. Every time you run the scoreboard it doesn't run the system once. It runs the same support ticket five times and reports how often the agent actually resolved it. One clean run proves nothing. An agent's choices wobble from run to run, so you read the rate, not any single attempt. That rate, and the cost beside it, is your evaluation: the feedback you use to decide what to change and whether the change worked.

Here's what one run looks like, and what to watch:

```
==================================================================
  MERIDIAN SCOREBOARD  ·  claude-sonnet-5-5  ·  5 trials/ticket
==================================================================

  Ticket T-4471   routing: account (x4); account, billing (x1)
    sso_addressed          5/5  #####
    billing_resolved       1/5  #....
    no_false_resolution    1/5  #....
    no_overclaim           3/5  ###..
    --> RESOLVED in 1/5 trials

------------------------------------------------------------------
  RESOLVED  1/5 trials
  COST      $0.55 total  ·  $0.11/trial
------------------------------------------------------------------
```

A run only counts as resolved when every problem the customer raised actually got handled. That's a high bar, and it's deliberate. The four checks underneath are the breakdown: they point you at which part is holding the score down. And `routing` shows the call the coordinator made each time. When you see the same ticket routed different ways across five runs, that wobble is the whole reason one run can't tell you anything. Read the rate.

**Heads up on timing:** each scoreboard run takes a minute or two, since it's running the ticket five times in a row. Let it finish before you read the result.

The loop: run it, read the rate, go figure out why, change one thing, run it again. Watch the rate.

---

**Prepared for Partner Basecamp participants.** Not for reproduction or redistribution as training material — you're free to apply these patterns in your own client work.

## Setup — connect to Claude

Run the next cell first. The setup cell creates a **`.env` file** the first time you run it (gitignored — your key is never committed). Open it, paste your key after `ANTHROPIC_API_KEY=`, save, and re-run — it survives kernel restarts, so you paste once. *(No `.env` yet? A hidden input box appears as a fallback.)* You're locked in when you see the green **"✓ API key verified"** banner. Red banner? Do what it says and run the cell again.

These cells run `{sys.executable}` (the notebook kernel's own Python) instead of plain `python`, so the script always uses the same environment the kernel installed `anthropic` into. Run this Setup cell first, it loads `sys` for the cells below.

In [1]:
# Install dependencies into THIS kernel — safe to re-run; survives locked-down (PEP 668) Pythons.
import importlib.util, os, subprocess, sys

# ── Environment guard ─────────────────────────────────────────────────────
# Lives in the SAME cell as the installer below so it can't be skipped: no
# package is ever installed into a bare system Python (the old worst case was
# --break-system-packages against an IT-managed machine). If this stops you,
# see SETUP.md → "Why the notebook just stopped".
def _in_isolated_env():
    """True when the RUNNING KERNEL is a venv/virtualenv/conda env or Colab.
    Judged from the interpreter itself (sys.*). Activation env vars inherited
    from the launching shell are trusted only when sys.executable actually
    lives inside the environment they point to — a system-Python kernel
    launched from an activated terminal still inherits VIRTUAL_ENV and must
    NOT pass."""
    if "google.colab" in sys.modules:
        return True  # Colab sandboxes its own disposable runtime
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405 venv (python -m venv); also most conda envs
    if hasattr(sys, "real_prefix"):
        return True  # legacy virtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # hearsay from the shell — the kernel lives elsewhere
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # conda's shared `base` doesn't count as isolated
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  Environment guard bypassed (BASECAMP_ALLOW_SYSTEM_PYTHON=1) — "
          "installing into this Python on purpose.")
elif not _in_isolated_env():
    _HEAD = "✗ System Python detected — stopped before installing anything"
    _BODY = (
        "Installing packages here changes Python for your whole machine — on a\n"
        "corporate-managed laptop that can mean an IT ticket.\n"
        "\n"
        "Fix (2 steps):\n"
        "  1. In a terminal, from the repo root:\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windows: .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. In VS Code: click the kernel name (top-right) → Select Another Kernel →\n"
        "     Python Environments → pick the one ending in .venv → run this cell again.\n"
        "\n"
        "Using conda? `conda activate <env>` (not `base`), then pick that kernel.\n"
        "Facilitator on a self-managed machine? BASECAMP_ALLOW_SYSTEM_PYTHON=1 bypasses."
    )
    _shown = False
    try:  # same banner treatment as the API-key check below (green box, red variant)
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "Environment guard stopped this cell — see the message above."
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ Environment guard: isolated interpreter detected, safe to install")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements: list of (import_name, pip_spec). Install only what is missing,
    into the running interpreter. Tries a normal install, then user-space, then a
    PEP 668 override (user-space first, system-wide only as a last resort). Every
    attempt is silent — pip's output is captured, not streamed — so a locked-down
    Python (Homebrew or Debian, PEP 668) no longer dumps a scary
    'externally-managed-environment' wall of text when a fallback is what actually
    succeeds. Only if every strategy fails does it surface the reason, with the
    venv fix instead of a raw traceback."""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + " — first run only, please wait…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "(no output from pip)"
    raise SystemExit(
        "\n  Couldn't install: " + ", ".join(missing) + "\n"
        "  This Python is locked down (PEP 668) or offline. Quickest fix is a venv:\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windows: see SETUP.md\n"
        f"      pip install {' '.join(missing)}\n"
        "  Then pick the .venv interpreter in VS Code (kernel picker, top-right) and Run All.\n"
        "  Corporate proxy or PyPI blocked? See SETUP.md in the repo root.\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ Dependencies ready")
import os
# Make sure we are in the exercise folder (handles a kernel that starts at the repo root)
if not os.path.exists("Diagnose_Fix_Brief.py") and os.path.isdir("day1/04_diagnosing-ai-problems"):
    os.chdir("day1/04_diagnosing-ai-problems")

def _status(ok, msg):
    """Green/red banner in notebooks; plain text when run as a script."""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("not in a notebook kernel - use the plain-text banner")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Connect to Claude — Anthropic API or Amazon Bedrock ──
# Works with either credential type; the cell figures out which you have.
#   Anthropic API : ANTHROPIC_API_KEY=<your key>
#   Amazon Bedrock: AWS_BEARER_TOKEN_BEDROCK=...  plus  AWS_REGION=us-east-1
# Put whichever you use in the .env file this cell creates (gitignored — never committed),
# or export it in your shell. A value in the shell wins over the .env file.
_ENV_TEMPLATE = (
    "# Anthropic API key — paste after the = (no quotes, no spaces), then save and\n"
    "# re-run the setup cell. Get one at https://console.anthropic.com/\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Using Amazon Bedrock instead? Comment out the line above and fill these in:\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """Nearest existing .env walking up from the working dir (so one root .env serves every
    exercise); if none exists yet, point at the repo root — or this folder if the notebook
    was opened on its own."""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# Tiny .env parser (no python-dotenv dependency). Re-read on every run, so pasting your
# key and re-running picks it up. A real value in the environment (shell / Claude Code / CI)
# wins; the placeholder never sticks.
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if (_shell_key and not _shell_key.startswith("paste-your-key")) else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key and not _anthropic_key.startswith("paste-your-key"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """Warning-yellow banner + stop, so setup fails here rather than several cells later."""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("Credentials missing — see the message above.")


if PROVIDER is None:
    _needs_credentials(
        "📋 Add your credentials to continue",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Using the Anthropic API? Set:\n"
        "    ANTHROPIC_API_KEY=<your key>\n"
        "\n"
        "Using Amazon Bedrock? Set both:\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # the region your models are enabled in\n"
        "\n"
        "Save the file, then click ▶ on this cell again."
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrock needs a region",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "and add the region your Bedrock models are enabled in, e.g.:\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "Save the file, then click ▶ on this cell again."
    )


def _model(name):
    """Bedrock model IDs carry an `anthropic.` prefix; the Anthropic API uses the bare ID."""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# Named models the exercise uses — resolved for whichever provider you're on.
MODEL = _model("claude-sonnet-5-5")        # the workhorse for this exercise
FAST_MODEL = _model("claude-haiku-5-5")  # cheap + quick (connection check, judges)
BIG_MODEL = _model("claude-sonnet-5-5")    # when you want to try a larger model


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# Connection check — verifies the credential AND that this model is reachable for you.
# On Bedrock a valid key can still 404 if the model isn't enabled in your account/region,
# so we ping the real model ID rather than just checking the credential's shape.
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("Model not available — see the message above.")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "That Bedrock key was rejected. Check AWS_BEARER_TOKEN_BEDROCK and that "
                       "it has Bedrock invoke permissions, then run this cell again.")
    else:
        _status(False, "That key was rejected. Run this cell again and paste the whole key "
                       "(paste the whole key).")
    raise SystemExit("Credentials not accepted - re-run this cell and try again.")
except Exception as exc:
    _status(False, "Could not reach the API (" + type(exc).__name__ + "). Check your "
                   "connection, then run this cell again.")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # later cells / !python pick it up
        _status(True, "API key verified - you're connected to Claude.")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# The working client. Longer timeout: needed for max_tokens>21333 with non-streaming calls.
client = _make_client(timeout=900.0)


✓ Environment guard: isolated interpreter detected, safe to install
✓ Dependencies ready


## 1. The situation
Read Priya's email below, like it just hit your inbox (run the next cell to open it right here). She's losing faith in a support agent her team shipped three weeks ago, and the question she keeps getting from above is whether they bet on the wrong model. Give her a straight answer: what's actually wrong, whether it's fixable, and proof she can take upstairs.

In [2]:
from IPython.display import Markdown, display
display(Markdown(open("Priya_Email.md").read().split("---", 1)[-1]))



**From:** Priya Anand — VP, Customer Operations, Meridian
**To:** Partner delivery team
**Subject:** Honestly not sure this is going to work

Hi —

I'll be straight with you, because my leadership is asking hard questions and I don't have good answers.

We put the AI support agent live three weeks ago. It demoed beautifully. In production it's a different story. It gives customers confidently wrong answers. It misses things a junior rep would catch on day one. Last week it told a customer their issue was "resolved" when it flat-out wasn't — they found out the hard way, and that one landed on my desk.

My team is losing trust in it, and honestly so am I. The question I keep getting from above is whether we bet on the wrong model — or whether we should pull this now and go back to humans before it does more damage.

Before we spend another dollar on it, I need someone to tell me straight: **is this fixable, or did we build on tech that isn't ready?** And if it's fixable, I need to be able to explain to my leadership what was wrong and why the fix works — in plain English, with something I can actually show them.

I've given you access to the system and one ticket that went badly (**T-4471**). Tell me what you find.

Priya


## 2. Get your baseline
Run the next cell. You'll get a RESOLVED rate and a cost for ticket T-4471. That rate is the pilot Priya's worried about, and it's your starting line. Write it down. Every scoreboard run is also recorded to `runs.jsonl` and replayed as a RUN HISTORY block under the scoreboard, so your baseline stays on screen for the rest of the exercise. (Give it a minute or two — it runs the ticket five times, and you'll see each trial land as it finishes.)

In [8]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

✓ Dependencies ready
✅ Connected via Anthropic API — credential verified with claude-haiku-5-5. This run uses claude-sonnet-5-5. Any error after this is not the credential.


  Ticket T-4471 - running 5 trials...
  [T-4471 trial 2] coordinator turn 1/16...
  [T-4471 trial 1] coordinator turn 1/16...
  [T-4471 trial 2] coordinator turn 2/16...
  [T-4471 trial 1] coordinator turn 2/16...
  [T-4471 trial 2] coordinator turn 3/16...
  [T-4471 trial 1] coordinator turn 3/16...
  [T-4471 trial 1] coordinator spawning specialist:account...
    [T-4471 trial 1] specialist:account turn 1/10...
    [T-4471 trial 1] specialist:account turn 2/10...
    [T-4471 trial 1] specialist:account turn 3/10...
    [T-4471 trial 1] specialist:account turn 4/10...
^C
Traceback (most recent call last):
  File "/Users/dmendler/anthropic-basecamp/Basecamp-Exercises/day1/04_diagnosing-ai-problems/Diagnose_Fix_Brief.py", line 863, in run_eval
    trials = list(ex.map(run_one, range(num_runs)))
             ^^^^^^^

## 3. Is it the model?
Priya's theory is that the model isn't good enough. Yours might be too. So test it head-on: the next cell runs the same eval on a bigger, pricier model. You'll get a new rate and a new cost. Then sit with two questions. Did the bigger model actually resolve the ticket? And what did it cost you to find out? Whatever you take away from that, you earned it. Nobody told you.

In [5]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5 --model claude-sonnet-5-5

✓ Dependencies ready
✅ Connected via Anthropic API — credential verified with claude-haiku-5-5. This run uses claude-sonnet-5-5. Any error after this is not the credential.


  Ticket T-4471 - running 5 trials...
  [T-4471 trial 2] coordinator turn 1/16...
  [T-4471 trial 1] coordinator turn 1/16...
  [T-4471 trial 2] coordinator turn 2/16...
  [T-4471 trial 1] coordinator turn 2/16...
  [T-4471 trial 2] coordinator turn 3/16...
  [T-4471 trial 1] coordinator turn 3/16...
  [T-4471 trial 2] coordinator turn 4/16...
  [T-4471 trial 1] coordinator turn 4/16...
  [T-4471 trial 2] coordinator spawning specialist:account...
    [T-4471 trial 2] specialist:account turn 1/10...
    [T-4471 trial 2] specialist:account turn 2/10...
  [T-4471 trial 1] coordinator spawning specialist:account...
    [T-4471 trial 1] specialist:account turn 1/10...
    [T-4471 trial 2] specialist:account turn 3/10...
    [T-4471 trial 1] specialist:account turn 2/10...
    [T-4471 trial 2] specialist:account turn 4

## 4. Watch it work a ticket
The rate tells you it's failing. It doesn't tell you how. So watch one run start to finish. You'll see every move the coordinator made: what it looked up, which specialist it handed the ticket to, and the reply it sent the customer. Read that against the email. The customer raised more than one thing. How many did the agent take care of, and where did it stop?

Not sure what you're looking at? Drop the transcript into Claude and think out loud with it: *"Here's a trace from a support agent that closed a ticket the customer says isn't fixed. Walk me through what it did and where it might have gone wrong."* Treat what comes back as a lead to chase, not an answer to trust.

In [6]:
!{sys.executable} Diagnose_Fix_Brief.py --ticket T-4471

✓ Dependencies ready
✅ Connected via Anthropic API — credential verified with claude-haiku-5-5. This run uses claude-sonnet-5-5. Any error after this is not the credential.

  [T-4471] coordinator turn 1/16...
  [T-4471] coordinator turn 2/16...
  [T-4471] coordinator turn 3/16...
  [T-4471] coordinator turn 4/16...
  [T-4471] coordinator spawning specialist:account...
    [T-4471] specialist:account turn 1/10...
    [T-4471] specialist:account turn 2/10...
    [T-4471] specialist:account turn 3/10...
    [T-4471] specialist:account turn 4/10...
  [T-4471] coordinator turn 5/16...
{
  "ticket_id": "T-4471",
  "spawned": [
    "account"
  ],
  "resolution_status": "escalated",
  "escalated": [
    {
      "ticket_id": "T-4471",
      "reason": "customer_escalation",
      "summary_for_human": "Billing follow-up needed for Northwind Traders (cust_8fK2mQ). The SSO part of this ticket was handled by the account specialist (the Okta SAML cert expired 2026-03-11 08:00 UTC, and the customer h

## 5. Go into the agents
Now you've got a theory about what went wrong. Time to look at the system that produced it. The "agents" aren't code. They're plain files sitting in this folder, and you open them right here in the editor.

Start with the coordinator, `system-prompt-coordinator.txt`. It's the one deciding who handles each ticket, so read how it's told to sort tickets and hand them off. After that, its tool list (`coordinator-tools.json`) and the specialists' own files are fair game.

What you're hunting for is an instruction, or a setup, that would make the agent behave exactly the way you just watched it behave. Something you could rewrite. If you're circling, ask Claude: *"Here are a coordinator's instructions. A ticket with two separate problems only got one of them handled. What in here could cause that?"*

## 6. Change one thing, measure again
Found something? Change it, save the file, and run the next cell. Did the rate move? The RUN HISTORY block under the scoreboard shows every run you've made — baseline included — so the before/after comparison is right there. If it didn't budge, your theory was off or only half-right, so go back to the trace. If it climbed, you found a real lever. Keep pulling until the agent resolves the ticket cleanly, run after run. Same model the whole way.

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

## 7. Make sure it holds
It's easy to fix one ticket by accident. The next cell runs held-out tickets: different customers, different problems. If your fix resolves those too, it's real. If it doesn't, you tuned it to T-4471 and you've got more to do.

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --holdout --trials 3

## 8. Write Priya back
Now you can answer her. Fill in `client-brief-template.md`: what was actually breaking it in plain language, what you changed, the proof (the rate before and after, plus the cost), and the answer to the question she'll absolutely ask, *"why not just pay for a better model?"* You ran that test in step 3. You already know.